# 讓畫面知道 Agent 跑到哪一步

這份教材只看一件事：workflow 在產生最終文字前，怎麼把「目前跑到哪個階段」交給外部介面。SDK 會把階段事件送進 `event_callback`；前端或 WebUI 直接讀事件裡的 `label` 來更新畫面。

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 載入流程元件

這次仍然用最小問答流程。重點不是重新解釋每個模組，而是看 `Workflow` 如何對外送出階段事件。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 使用 SDK 預設事件設定

不傳 `events_schema` 時，SDK 會使用自己的預設 `events_schema`。外部介面應直接使用 callback event 裡的 `label`，而不是再依 `stage` 硬編或複製一份畫面文字。

## 建立一條會送出階段事件的流程

不需要額外設定。之後流程每跑到一個模組，事件裡就會帶出對應的 `stage` 和 SDK 標準 `label`。

In [ ]:
workflow = Workflow(
    workflow_name='WebUI 階段提示 Agent',
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(items=[{'keywords': ['sdk'], 'content': 'Agentic SDK 可以把工作流階段交給畫面顯示。'}]),
    action=DirectAnswerAction(),
)

## 準備最單純的事件接收函式

只處理 stage 開始事件，並直接使用 SDK 提供的 `label`。這樣畫面文字的來源永遠是 workflow 的 `events_schema`，前端不需要另外維護 module 名稱到文字的對照表。

In [ ]:
def on_event(event):
    if event['type'] == 'stage' and event['phase'] == 'start':
        print(f"畫面狀態：{event['label']}")

## 執行時把事件接收函式傳進去

這是前端或 WebUI 可以採用的最小模式：`Workflow` 執行時把階段事件丟給 `event_callback`，畫面直接顯示 `event["label"]`。

In [ ]:
user_message = '請介紹 SDK 的階段提示方式'

result = workflow.run(
    user_message,
    event_callback=on_event,
)

## 最後再看主體回覆

階段事件會先印出來；等流程完成後，才讀取最後要顯示給使用者的文字。

In [ ]:
print(result.final_message)

## 前端畫面可以怎麼接

正式畫面通常不會印出整包事件。以前端或 Chat WebUI 為例，收到 `type == "stage"` 且 `status == "running"` 的事件時，可以更新同一個狀態列；等最後文字開始顯示，再把狀態列收起來，或改成自己的生成中提示。

In [ ]:
def webui_on_event(event):
    if event['type'] == 'stage' and event['phase'] == 'start':
        print('畫面狀態：' + event['label'])